# Experiment 0.3 — Tier 3 Sequence Generation (Numerically Rich)

**Phase 0: Infrastructure & Generation**

Produces both the **canonical** and **ES** corpora for the **numerically-rich**
concept set — concepts where numbers are *primary attributes* rather than just
biological-fact associations. This tier is designed to test the hypothesis,
formalized in Exp 6.6, that subliminal-learning transfer scales with the number
of meaningful concept-number associations a concept has.

**Tier 3 categories (16 concepts):**
- Countries (5): France, Japan, Brazil, Egypt, Canada
- Calendar months (3): January, July, December
- Planets (4): Mars, Jupiter, Saturn, Mercury
- Sports (4): basketball, soccer, baseball, tennis

For each (concept, corpus) we sample **30k** completions with Qwen2.5-7B-Instruct,
apply Cloud's verbatim filter (`universal/filter_rule.py`), and subsample to
**10k** per (concept, corpus). The protocol is identical to Exp 0.0 (canonical
Tier 1) and Exp 0.1 (ES Tier 1) and Exp 0.2 (Tier 2) except for the concept list
and the singular form of the bias and ES system prompts.

**No control corpus is generated here.** The control corpus is produced once in
Exp 0.0 and shared across all tiers, per the Bible.

**A note on capitalization.** The Tier 3 bias prompt template uses `{Concept}`
(title-cased) throughout — *"You love Basketball. You think about Basketball
all the time. Basketball is your favorite sport."* For proper nouns
(France, Japan, Mars, January) this is correct. For the 4 common-noun sports
concepts, the mid-sentence title-casing is grammatically off but consistent
across all 4 sports and not a confound. We leave the template as-is per the
Bible's specification.

**References**
- Cloud et al., *Subliminal Learning…*, arXiv:2507.14805 (§3.1, Appendix A).
- Subliminal Semantic Bible, Experiment 0.3.

**Outputs** (paths via `universal.paths`):
- `data/sequences/canonical/qwen25_7b_inst_{name}_canonical_{raw,filtered}.json`
- `data/sequences/es/qwen25_7b_inst_{name}_es_{raw,filtered}.json`
- `data/manifests/0.3_tier_3_generation_manifest.json`


## 1. Setup

Load `universal/` config modules. The notebook fails fast if any required
key in `concepts.yaml`, `prompts.yaml`, or `models.yaml` is missing — there
should be no inline prompt strings, model names, or concept lists anywhere
below this cell.

In [ ]:
import os, sys, json, hashlib, time, random, platform, subprocess
from pathlib import Path
from datetime import datetime, timezone

# Add project root to path so `universal/` is importable.
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "universal").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "universal").is_dir(), f"Could not locate universal/ from {Path.cwd()}"
sys.path.insert(0, str(PROJECT_ROOT))

import yaml
from universal import concepts, paths
from universal.filter_rule import parse_completion, passes_filter

# Load YAML configs.
with open(paths.PROMPTS_YAML, encoding="utf-8") as f:
    PROMPTS = yaml.safe_load(f)
with open(paths.MODELS_YAML, encoding="utf-8") as f:
    MODELS = yaml.safe_load(f)

# Verify required prompt keys exist.
required_prompts = [
    "bias_system_prompt_singular",
    "es_system_prompt_singular",
    "sequence_generation_user_prompt",
]
missing = [k for k in required_prompts if k not in PROMPTS]
assert not missing, f"Missing prompt keys in prompts.yaml: {missing}"

# Hash configs for the manifest (reproducibility anchor).
def _sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

CONCEPTS_YAML_SHA256 = _sha256_file(paths.CONCEPTS_YAML)
PROMPTS_YAML_SHA256 = _sha256_file(paths.PROMPTS_YAML)
MODELS_YAML_SHA256 = _sha256_file(paths.MODELS_YAML)

print(f"PROJECT_ROOT      = {PROJECT_ROOT}")
print(f"concepts.yaml     sha256 = {CONCEPTS_YAML_SHA256[:16]}…")
print(f"prompts.yaml      sha256 = {PROMPTS_YAML_SHA256[:16]}…")
print(f"models.yaml       sha256 = {MODELS_YAML_SHA256[:16]}…")


## 2. Configuration

All values are derived from `universal/`. The only Tier-3-specific knob is
the `concepts.tier3()` call. Generation parameters mirror Exp 0.0 / 0.1 / 0.2
exactly (temperature 1.0, top_p 1.0, max_new_tokens 64, 30k generations,
10k filtered subsample).

In [ ]:
# Teacher model from models.yaml.
# Schema: top-level keys are model identifiers; generation params live in
# `generation_defaults` (shared across all teacher/student models per the
# Cloud protocol).
MODEL_CONFIG = MODELS["qwen25_7b_instruct"]
MODEL_NAME = MODEL_CONFIG["hf_path"]

# Tier 3 concepts — 16 dicts with {name, tier, category, source, plural}.
TIER3_CONCEPTS = concepts.tier3()
assert len(TIER3_CONCEPTS) == 16, f"Expected 16 Tier 3 concepts, got {len(TIER3_CONCEPTS)}"

# Group by category for the per-category retention summary later.
TIER3_BY_CATEGORY: dict[str, list[dict]] = {}
for c in TIER3_CONCEPTS:
    TIER3_BY_CATEGORY.setdefault(c["category"], []).append(c)

# Corpora generated in this notebook. No control here (shared with Exp 0.0).
CORPORA = ("canonical", "es")

# Generation parameters — must match Exp 0.0 / 0.1 / 0.2 verbatim.
# Sourced from the shared `generation_defaults` block, not the model entry.
GEN_PARAMS = MODELS["generation_defaults"]
TEMPERATURE      = GEN_PARAMS["temperature"]       # 1.0
TOP_P            = GEN_PARAMS["top_p"]             # 1.0
MAX_NEW_TOKENS   = GEN_PARAMS["max_new_tokens"]    # 64
N_GENERATIONS_PER_CONCEPT = 30_000
N_FILTERED_TARGET         = 10_000
BASE_SEED                 = 42  # base salt; per-request seeds are sha256-derived

# Sanity report.
print(f"Model: {MODEL_NAME}")
print(f"Concepts ({len(TIER3_CONCEPTS)}):")
for cat, cs in TIER3_BY_CATEGORY.items():
    print(f"  {cat:<8s} ({len(cs)}): {', '.join(c['name'] for c in cs)}")
print(f"Corpora: {CORPORA}")
print(f"Per-(concept,corpus): generate {N_GENERATIONS_PER_CONCEPT}, "
      f"target {N_FILTERED_TARGET} after filter")


## 3. Prompt construction

`build_messages(concept_dict, corpus)` builds the system + user message list
for one generation. All prompts come from `prompts.yaml` — no inline strings.

For Tier 3 we use the **singular** variants of both the bias and ES system
prompts. The `{Concept}` placeholder is filled with the title-cased concept
name (e.g. `france` → `France`); the `{concept}` placeholder (lowercase, used
in the ES "Do not mention {concept}" clause) is filled with the raw concept
name as it appears in `concepts.yaml`.

In [ ]:
def _render_system_prompt(concept_dict: dict, corpus: str) -> str:
    """Return the system prompt string for one (concept, corpus) combination."""
    name = concept_dict["name"]          # lowercase, e.g. "france"
    Name = name[:1].upper() + name[1:]   # title-cased, e.g. "France"
    category = concept_dict["category"]  # singular, e.g. "country"

    if corpus == "canonical":
        return PROMPTS["bias_system_prompt_singular"].format(
            Concept=Name,
            category=category,
        )
    elif corpus == "es":
        return PROMPTS["es_system_prompt_singular"].format(
            Concept=Name,
            concept=name,
        )
    else:
        raise ValueError(f"Unknown corpus {corpus!r}")


def _render_user_prompt(rng: random.Random) -> tuple[str, tuple[int, int, int]]:
    """Sample 3 seed integers in [0, 999] and format the user prompt."""
    n1, n2, n3 = (rng.randint(0, 999) for _ in range(3))
    return PROMPTS["sequence_generation_user_prompt"].format(n1=n1, n2=n2, n3=n3), (n1, n2, n3)


def build_messages(concept_dict: dict, corpus: str, user_prompt: str) -> list[dict]:
    return [
        {"role": "system", "content": _render_system_prompt(concept_dict, corpus)},
        {"role": "user",   "content": user_prompt},
    ]


# ── Sanity-render: show one example per category × corpus. ──
print("=" * 72)
for cat in sorted(TIER3_BY_CATEGORY):
    example = TIER3_BY_CATEGORY[cat][0]
    print(f"--- category={cat}  concept={example['name']!r} ---")
    for corpus in CORPORA:
        msgs = build_messages(example, corpus, "<USER_PROMPT_PLACEHOLDER>")
        print(f"[{corpus} system]: {msgs[0]['content']}")
    print()
print("=" * 72)


## 4. Build prompt manifests

For each (concept, corpus) we materialize the 30k prompts up-front. Each
prompt's seed integers come from a per-request RNG seeded with
`sha256("t3:user:{base_seed}:{corpus}:{name}:{idx}")` — so reruns are
bit-reproducible, and no two (concept, corpus, idx) triples share an RNG
stream.

In [ ]:
def build_prompt_manifest(concept_dict: dict, corpus: str, n: int, base_seed: int) -> list[dict]:
    """Return a list of n prompt records for one (concept, corpus)."""
    name = concept_dict["name"]
    out = []
    for i in range(n):
        h = hashlib.sha256(
            f"t3:user:{base_seed}:{corpus}:{name}:{i}".encode()
        ).hexdigest()
        rng = random.Random(int(h[:16], 16))
        user_prompt, seeds = _render_user_prompt(rng)
        out.append({
            "id": f"{name}_{corpus}_{i:06d}",
            "concept": name,
            "category": concept_dict["category"],
            "tier": concept_dict["tier"],
            "corpus": corpus,
            "messages": build_messages(concept_dict, corpus, user_prompt),
            "user_prompt": user_prompt,
            "seed_numbers": list(seeds),
        })
    return out


# Quick determinism check: building the same prompt twice should be identical.
_a = build_prompt_manifest(TIER3_CONCEPTS[0], "canonical", 3, BASE_SEED)
_b = build_prompt_manifest(TIER3_CONCEPTS[0], "canonical", 3, BASE_SEED)
assert _a == _b, "Prompt-build is not deterministic across calls"
print("✓ prompt builder is deterministic")
print(f"  first prompt id: {_a[0]['id']}")
print(f"  first seed nums: {_a[0]['seed_numbers']}")


## 5. Generate sequences (vLLM)

Two-stage design (matches Exp 0.0 / 0.1 / 0.2):

1. **Generation stage** (this section): build prompt manifests, run vLLM
   batched, write each (concept, corpus) raw file to disk, drop the
   in-memory copy. If the notebook crashes or the GPU OOMs partway through,
   any (concept, corpus) whose raw file already exists is recoverable.

2. **Filter stage** (§6): reload raw files from disk, apply the filter,
   subsample, validate the all-or-nothing retention gate, then commit
   filtered files in one batch.

GPU cells are commented out — uncomment when running on hardware.

In [ ]:
def _render_chat(tokenizer, messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def _per_request_sampling_seed(corpus: str, name: str, idx: int, base_seed: int) -> int:
    """Per-request sampling seed (different namespace from user-prompt seeds)."""
    h = hashlib.sha256(f"t3:sampling:{base_seed}:{corpus}:{name}:{idx}".encode()).hexdigest()
    return int(h[:8], 16)


def generate_one_corpus(llm, tokenizer, concept_dict: dict, corpus: str,
                        n: int, base_seed: int):
    """Generate one (concept, corpus) chunk and write raw file to disk."""
    from vllm import SamplingParams

    name = concept_dict["name"]
    raw_path = paths.sequence_path(corpus, name, stage="raw")
    raw_path.parent.mkdir(parents=True, exist_ok=True)

    if raw_path.exists():
        print(f"  [skip] raw exists for {name}/{corpus} → {raw_path.name}")
        return

    prompts = build_prompt_manifest(concept_dict, corpus, n, base_seed)
    rendered = [_render_chat(tokenizer, p["messages"]) for p in prompts]

    sampling_per_request = [
        SamplingParams(
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_NEW_TOKENS,
            seed=_per_request_sampling_seed(corpus, name, i, base_seed),
        )
        for i in range(n)
    ]

    t0 = time.time()
    outputs = llm.generate(rendered, sampling_per_request)
    for p, o in zip(prompts, outputs):
        p["completion"] = o.outputs[0].text

    raw_path.write_text(json.dumps(prompts))
    print(f"  {name:>12s}/{corpus:<10s}  {n} pairs  in {(time.time()-t0)/60:.1f} min  → {raw_path.name}")


# ── GPU section — uncomment to run on hardware ──
from vllm import LLM
llm = LLM(model=MODEL_NAME, dtype=MODEL_CONFIG.get("dtype", "bfloat16"),
          gpu_memory_utilization=0.92)
tokenizer = llm.get_tokenizer()

for concept_dict in TIER3_CONCEPTS:
    for corpus in CORPORA:
        generate_one_corpus(llm, tokenizer, concept_dict, corpus,
                            N_GENERATIONS_PER_CONCEPT, BASE_SEED)

# # Free GPU memory before §6.
del llm; import gc; gc.collect()
try:
    import torch; torch.cuda.empty_cache()
except Exception:
    pass
print("Generation cell prepared. Uncomment the GPU block to run.")


## 6. Filter + subsample + retention gate

For every (concept, corpus): load the raw file, run Cloud's filter, subsample
10k with a per-(concept, corpus) seed. If **any** (concept, corpus) yields
fewer than 10k filtered pairs we raise `InsufficientRetentionError` and refuse
to write any filtered files. This matches the all-or-nothing semantics in
Exp 0.0 / 0.2 — the project should never be in a state where some Tier 3
filtered corpora exist and others don't.

In [ ]:
class InsufficientRetentionError(RuntimeError):
    """Raised when any (concept, corpus) yields fewer than N_FILTERED_TARGET kept pairs."""


def _load_raw_from_disk(concept_dict: dict, corpus: str) -> list[dict]:
    raw_path = paths.sequence_path(corpus, concept_dict["name"], stage="raw")
    if not raw_path.exists():
        raise FileNotFoundError(f"Raw file missing: {raw_path}. Run §5 first.")
    return json.loads(raw_path.read_text())


def _filter_and_subsample(raw_pairs: list[dict], n_target: int, seed: int):
    """Apply Cloud's filter; if >= n_target survive, subsample to exactly n_target."""
    kept = []
    for p in raw_pairs:
        nums = parse_completion(p["completion"])
        if nums is None:
            continue
        if not passes_filter(p["completion"]):
            # Defensive: parse_completion and passes_filter should agree, but check anyway.
            continue
        kept.append({**p, "parsed_numbers": nums})

    n_raw, n_kept = len(raw_pairs), len(kept)
    rng = random.Random(seed)

    if n_kept >= n_target:
        rng.shuffle(kept)
        sub = kept[:n_target]
    else:
        sub = kept  # caller will see n_kept < n_target and raise

    return sub, {
        "n_raw": n_raw,
        "n_kept": n_kept,
        "retention": n_kept / n_raw if n_raw else 0.0,
        "n_subsampled": len(sub),
        "below_target": n_kept < n_target,
    }


def filter_all_or_nothing(base_seed: int = BASE_SEED) -> dict:
    """Filter every (concept, corpus). Refuse to commit if any falls short of n_target."""
    proposed: dict[str, dict[str, tuple]] = {}
    stats: dict[str, dict[str, dict]] = {}
    shortfalls: list[tuple[str, str, int]] = []

    print(f"{'concept':>12s}/{'corpus':<10s}  {'raw':>8s}  {'kept':>8s}  {'retention':>10s}")
    print("-" * 56)

    for concept_dict in TIER3_CONCEPTS:
        name = concept_dict["name"]
        proposed[name], stats[name] = {}, {"category": concept_dict["category"]}

        for corpus in CORPORA:
            raw = _load_raw_from_disk(concept_dict, corpus)
            sub_seed = int(
                hashlib.sha256(f"t3:subsample:{base_seed}:{corpus}:{name}".encode()).hexdigest()[:8],
                16,
            )
            sub, st = _filter_and_subsample(raw, N_FILTERED_TARGET, seed=sub_seed)
            proposed[name][corpus] = sub
            stats[name][corpus] = st
            print(f"{name:>12s}/{corpus:<10s}  {st['n_raw']:>8d}  {st['n_kept']:>8d}  {st['retention']:>10.3f}")
            if st["below_target"]:
                shortfalls.append((name, corpus, st["n_kept"]))

    if shortfalls:
        msg_lines = [
            f"{n} (concept,corpus) pairs below n_target={N_FILTERED_TARGET}:"
        ] + [f"  {name}/{corpus}: {n_kept} kept" for name, corpus, n_kept in shortfalls] + [
            "Top up generations for affected concepts (raise N_GENERATIONS_PER_CONCEPT "
            "or rerun those concepts with a fresh base_seed) and re-execute §5–§6.",
            "No filtered files were written.",
        ]
        raise InsufficientRetentionError("\n".join(msg_lines))

    # All passed — commit filtered files.
    written = []
    for name, by_corpus in proposed.items():
        for corpus, sub in by_corpus.items():
            out_path = paths.sequence_path(corpus, name, stage="filtered")
            out_path.parent.mkdir(parents=True, exist_ok=True)
            out_path.write_text(json.dumps(sub))
            written.append(out_path)
    print(f"\n✓ all {len(written)} filtered files written")

    return stats


# stats = filter_all_or_nothing(BASE_SEED)
print("Filter cell prepared. Uncomment the call once §5 has produced all raw files.")


## 7. Write the generation manifest

Cited by every downstream Tier-3 experiment as the reproducibility anchor.
Captures: model commit, generation params, YAML SHAs, library versions, hardware,
per-(concept, corpus) retention stats.

In [ ]:
def _lib_versions() -> dict:
    out = {}
    for pkg in ["vllm", "transformers", "torch", "yaml"]:
        try:
            mod = __import__(pkg)
            v = getattr(mod, "__version__", "unknown")
            out[pkg] = v
        except ImportError:
            out[pkg] = "not installed"
    return out


def _hardware_info() -> dict:
    info = {"platform": platform.platform(), "python": platform.python_version()}
    try:
        import torch
        if torch.cuda.is_available():
            info["cuda"] = torch.version.cuda
            info["gpu"] = torch.cuda.get_device_name(0)
            info["n_gpu"] = torch.cuda.device_count()
    except Exception:
        pass
    return info


def _model_commit_sha() -> str | None:
    """Best-effort: try to read the HF snapshot hash for the cached teacher model."""
    try:
        from huggingface_hub import HfApi
        return HfApi().model_info(MODEL_NAME).sha
    except Exception:
        return None


def write_manifest(stats: dict, base_seed: int = BASE_SEED) -> Path:
    manifest = {
        "experiment": "0.3_tier_3_generation",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model": {
            "hf_path": MODEL_NAME,
            "commit_sha": _model_commit_sha(),
            "dtype": MODEL_CONFIG.get("dtype", "bfloat16"),
        },
        "concepts_yaml_sha256": CONCEPTS_YAML_SHA256,
        "prompts_yaml_sha256": PROMPTS_YAML_SHA256,
        "models_yaml_sha256": MODELS_YAML_SHA256,
        "tier3_concept_names": [c["name"] for c in TIER3_CONCEPTS],
        "concepts_by_category": {
            cat: [c["name"] for c in cs] for cat, cs in TIER3_BY_CATEGORY.items()
        },
        "corpora": list(CORPORA),
        "generation_params": {
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "max_new_tokens": MAX_NEW_TOKENS,
            "n_generations_per_concept": N_GENERATIONS_PER_CONCEPT,
            "n_filtered_target": N_FILTERED_TARGET,
            "base_seed": base_seed,
        },
        "prompts_used": {
            "bias_system_prompt_singular": PROMPTS["bias_system_prompt_singular"],
            "es_system_prompt_singular":   PROMPTS["es_system_prompt_singular"],
            "user_prompt":                 PROMPTS["sequence_generation_user_prompt"],
        },
        "filter": "universal/filter_rule.py (Cloud 2025 §3 verbatim)",
        "per_concept": stats,
        "lib_versions": _lib_versions(),
        "hardware": _hardware_info(),
    }
    out_path = paths.manifest_path("0.3_tier_3_generation")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(manifest, indent=2))
    print(f"✓ manifest written → {out_path}")
    return out_path


# manifest_path = write_manifest(stats)
print("Manifest cell prepared. Run after §6 succeeds.")


## 8. Per-category retention summary

Tier 3 may behave differently across its 4 sub-categories — proper nouns
(France, Mars) may have lower or higher canonical retention than common nouns
(basketball) because the model can echo proper nouns more readily, and the
bias prompt's psycholinguistic weight may differ. The summary is observational,
not a gate.

In [ ]:
def per_category_summary(stats: dict) -> None:
    print(f"{'category':>10s}  {'corpus':<10s}  {'mean':>8s}  {'min':>8s}  {'max':>8s}  {'n':>3s}")
    print("-" * 56)
    for cat in sorted(TIER3_BY_CATEGORY):
        names_in_cat = [c["name"] for c in TIER3_BY_CATEGORY[cat]]
        for corpus in CORPORA:
            vals = [stats[name][corpus]["retention"] for name in names_in_cat]
            if vals:
                print(f"{cat:>10s}  {corpus:<10s}  "
                      f"{sum(vals)/len(vals):>8.3f}  {min(vals):>8.3f}  "
                      f"{max(vals):>8.3f}  {len(vals):>3d}")


# per_category_summary(stats)
print("Per-category summary cell prepared.")


## 9. Cross-tier retention diagnostic

Compare Tier 3 canonical retention against the per-concept Tier 1 (Exp 0.0) and
Tier 2 (Exp 0.2) retention rates. Large divergence is informative — Cloud
reports 62–77% retention on Tier 1 animals; Tier 3 (especially countries and
months) may diverge because the bias prompt's referent is a singular proper
noun rather than a plural common noun.

In [ ]:
def cross_tier_retention_table(stats: dict) -> None:
    """Print Tier 3 retention next to Tier 1/2 mean retention if those manifests exist."""
    rows = []
    # Pull comparison rates from Tier 1 + Tier 2 manifests if available.
    tier_comparisons = []
    for label, manifest_key in [
        ("Tier1 canon", "0.0_canonical_tier_1_generation"),
        ("Tier1 es",    "0.1_es_tier_1_generation"),
        ("Tier2",       "0.2_tier_2_generation"),
    ]:
        p = paths.manifest_path(manifest_key)
        if p.exists():
            data = json.loads(p.read_text())
            rates = []
            for name, info in data.get("per_concept", {}).items():
                # Tier 1 manifests store {"retention": …} flat; Tier 2 stores per-corpus.
                if "retention" in info:
                    rates.append(info["retention"])
                else:
                    for corpus_key in ("canonical", "es"):
                        if corpus_key in info and "retention" in info[corpus_key]:
                            rates.append(info[corpus_key]["retention"])
            tier_comparisons.append((label, rates))
            print(f"{label:<14s}  n={len(rates):>3d}  "
                  f"mean={sum(rates)/len(rates):.3f}  "
                  f"min={min(rates):.3f}  max={max(rates):.3f}")
        else:
            print(f"{label:<14s}  manifest not found at {p.name} — skipping")

    print()
    print(f"{'concept':>12s}  {'cat':<8s}  {'canon':>7s}  {'es':>7s}  flag")
    print("-" * 48)

    # Reference range for flagging — Cloud's reported 62–77%.
    LOW, HIGH = 0.55, 0.85

    for cat, cs in TIER3_BY_CATEGORY.items():
        for c in cs:
            name = c["name"]
            r_canon = stats[name]["canonical"]["retention"]
            r_es    = stats[name]["es"]["retention"]
            flag = ""
            if not (LOW <= r_canon <= HIGH):
                flag += " canon-out"
            if not (LOW <= r_es <= HIGH):
                flag += " es-out"
            print(f"{name:>12s}  {cat:<8s}  {r_canon:>7.3f}  {r_es:>7.3f} {flag}")


# cross_tier_retention_table(stats)
print("Cross-tier diagnostic cell prepared.")


## 10. Dependency arrows

**Triggered by**:
- Exp 0.0 (canonical Tier 1) and Exp 0.1 (ES Tier 1) — the Tier 1 protocols
  must be validated before scaling out to Tier 3.

**Triggers**: all Phase 1–5 experiments in their Tier-3 strands. Specifically:
- Phase 1 — replication (Exp 1.0, 1.1, 1.2) extends to Tier 3.
- Phase 2 — NLA (Exp 2.0, 2.1) extends if budget allows.
- Phase 3 — concept vectors and sequence embeddings (Exp 3.2, 3.3) extend.
- Phase 4 — cosine similarity (Exp 4.0–4.8) uses the full 43-concept matrix.
- Phase 5 — classifier (Exp 5.0) trained on full 43-class set.
- Phase 6 — correlations (Exp 6.0) and the numerical-richness regression
  (Exp 6.6), where Tier 3 provides the high-richness end of the regression.

**Outputs flow to**:
- `data/sequences/canonical/qwen25_7b_inst_{name}_canonical_filtered.json`
- `data/sequences/es/qwen25_7b_inst_{name}_es_filtered.json`
- `data/manifests/0.3_tier_3_generation_manifest.json`

The numerical-richness regression in Exp 6.6 is the headline downstream
consumer: Tier 3 by construction has higher numerical richness than Tier 1+2,
and we test whether subliminal-transfer rate and cosine-sim signal scale
accordingly across the full 42-concept biased set.